# Vietnamese Clickbait Detector — Testing & Inference Notebook
Load a fine-tuned checkpoint and run evaluation, interactive prediction, or batch inference.

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes trl \
    accelerate scikit-learn pandas pyyaml pydantic tqdm matplotlib seaborn
print('Done.')

In [ ]:
import os, sys
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/clickbait-detector'  # <-- change this
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

CHECKPOINT_DIR = '/content/drive/MyDrive/clickbait_checkpoints/best'  # <-- change this
print('Project dir:', PROJECT_DIR)
print('Checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
from src.config import get_config
from src.model import load_finetuned_model

config = get_config('configs/config.yaml', profile='colab_free')
model, tokenizer = load_finetuned_model(config, checkpoint_path=CHECKPOINT_DIR)
print('Model loaded.')

## 2. Full Test Set Evaluation

In [ ]:
import pandas as pd
from src.data_loader import ClickbaitDataset, load_splits
from src.evaluate import full_evaluation

_, _, test_df = load_splits(config.data.processed_dir)
test_ds = ClickbaitDataset(test_df, tokenizer, config)

metrics = full_evaluation(model, tokenizer, test_ds, config)
print('\nTest Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
from IPython.display import Image, display
import os

results_dir = os.path.join(config.training.output_dir, 'results')
for fname in ['confusion_matrix.png', 'confidence_histogram.png']:
    p = os.path.join(results_dir, fname)
    if os.path.exists(p):
        print(fname)
        display(Image(p))

## 3. Interactive Single Prediction

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from src.predict import predict_single

title_input = widgets.Textarea(
    value='Sốc: Bí mật kinh hoàng đằng sau vụ việc này',
    description='Title:',
    layout=widgets.Layout(width='80%', height='60px'),
)
para_input = widgets.Textarea(
    value='Bạn sẽ không tin được những gì đã xảy ra trong vụ việc này...',
    description='Lead:',
    layout=widgets.Layout(width='80%', height='80px'),
)
btn = widgets.Button(description='Predict', button_style='primary')
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        result = predict_single(
            title_input.value, para_input.value, model, tokenizer, config
        )
        color = 'red' if result['label'] == 'clickbait' else 'green'
        print(f"Label:      {result['label']}")
        print(f"Confidence: {result['confidence']:.4f}")

btn.on_click(on_click)
display(title_input, para_input, btn, output)

## 4. Batch Prediction from CSV

In [ ]:
from google.colab import files
from src.predict import predict_batch
import os

print('Upload your CSV file:')
uploaded = files.upload()
input_csv = list(uploaded.keys())[0]

os.makedirs('outputs/predictions', exist_ok=True)
out_df = predict_batch(input_csv, model, tokenizer, config)
out_csv = 'outputs/predictions/batch_predictions.csv'
out_df.to_csv(out_csv, index=False)
print(f'Saved {len(out_df)} predictions to {out_csv}')
out_df[['title', 'pred_label', 'confidence']].head(10)

In [ ]:
from google.colab import files
files.download(out_csv)

## 5. Error Analysis

In [ ]:
from src.evaluate import error_analysis

errors = error_analysis(model, tokenizer, test_ds, config)
print(f'Misclassified: {len(errors)} / {len(test_ds)}')
errors[['title', 'true_label', 'pred_label', 'confidence']].head(20)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (label_name, group) in zip(axes, errors.groupby('true_label')):
    ax.hist(group['confidence'], bins=15, color='salmon', edgecolor='black')
    ax.set_title(f'True label: {label_name}')
    ax.set_xlabel('Confidence (P(clickbait))')
    ax.set_ylabel('Count')

plt.suptitle('Confidence Distribution of Misclassified Samples')
plt.tight_layout()
plt.show()